In [1]:
import numpy as np
import os

eps = 1e-8
kmax = 1000

In [4]:
def horner_value(a, v):
    b = a[0]
    for i in range(1, len(a)):
        b = a[i] + b * v
    return b


def horner_derivatives(a, v):
    # Derivate cu schema Horner extinsa:
    # c -> P'(v), d -> P''(v)
    b = a[0]
    c = 0.0
    d = 0.0
    for i in range(1, len(a)):
        d = d * v + 2.0 * c
        c = c * v + b
        b = a[i] + b * v
    return c, d

In [5]:
def newton(a, x0):
    x = float(x0)
    for k in range(kmax):
        Pv = horner_value(a, x)
        dPv, _ = horner_derivatives(a, x)
        if abs(dPv) <= eps:
            raise ValueError("|P'(x)| <= eps")
        # delta_x = P(xk) / P'(xk),  x_{k+1} = xk - delta_x
        delta_x = Pv / dPv
        x -= delta_x
        if abs(delta_x) >= 1e8:
            raise ValueError("divergenta")
        if abs(delta_x) < eps:
            return x, k + 1
    raise ValueError("nu a convergat")


def olver(a, x0):
    x = float(x0)
    for k in range(kmax):
        Pv = horner_value(a, x)
        dPv, d2Pv = horner_derivatives(a, x)
        if abs(dPv) <= eps:
            raise ValueError("|P'(x)| <= eps")
        #  ck = [P(xk)]^2 * P''(xk) / [P'(xk)]^3
        ck = (Pv**2 * d2Pv) / (dPv**3)
        # delta_x = P(xk)/P'(xk) + ck/2,  x_{k+1} = xk - delta_x
        delta_x = Pv / dPv + 0.5 * ck
        x -= delta_x
        if abs(delta_x) >= 1e8:
            raise ValueError("divergenta")
        if abs(delta_x) < eps:
            return x, k + 1
    raise ValueError("nu a convergat")

In [12]:
def collect(method, a, starts):
    roots = []
    for x0 in starts:
        try:
            roots.append(method(a, x0))
        except ValueError:
            pass

    roots.sort(key=lambda t: t[0])
    merged = []
    for r, s in roots:
        if not merged or abs(r - merged[-1][0]) >= 1e-5:
            merged.append((r, s))
        elif s < merged[-1][1]:
            merged[-1] = (r, s)
    return merged


def run(label, a, filename="roots.txt"):
    #  A = max|ai|, i=1..n;  R = (|a0| + A) / |a0|
    A = max(abs(ai) for ai in a[1:])
    R = (abs(a[0]) + A) / abs(a[0])

    starts = np.linspace(-R, R, 300)
    roots_newton = collect(newton, a, starts)
    roots_olver = collect(olver, a, starts)

    print(f"Interval: [-{R:.4f}, {R:.4f}]")

    print(f"Newton:")
    for r, s in roots_newton:
        print(f"    x* = {r:+.8f}  ({s} pasi)")

    print(f"Olver:")
    for r, s in roots_olver:
        print(f"    x* = {r:+.8f}  ({s} pasi)")

    print("Comparatie (radacini comune):")
    for rn, sn in roots_newton:
        for ro, so in roots_olver:
            if abs(rn - ro) < 1e-5:
                print(f"    x* ~ {rn:+.6f}   Newton: {sn} pasi | Olver: {so} pasi")
                break

    # scriere fisier: radacini distincte (|v1 - v2| > eps) din ambele metode
    all_roots = sorted(set([round(r, 5) for r, _ in roots_newton + roots_olver]))
    distinct = [all_roots[0]]
    for r in all_roots[1:]:
        if abs(r - distinct[-1]) > eps:
            distinct.append(r)

    with open(filename, "a") as f:
        f.write(f"=== {label} ===\n")
        f.write(f"Interval: [-{R:.4f}, {R:.4f}]\n")
        f.write(
            "Radacini distincte: " + ", ".join(f"{r:+.8f}" for r in distinct) + "\n\n"
        )

    print()

In [13]:
if os.path.exists("roots.txt"):
    os.remove("roots.txt")

run("P = (x-1)(x-2)(x-3)", a=[1.0, -6.0, 11.0, -6.0])
run("P = (x-2/3)(x-1/7)(x+1)(x-3/2)", a=[42.0, -55.0, -42.0, 49.0, -6.0])
run("P = (x-1)(x-1/2)(x-3)(x-1/4)", a=[8.0, -38.0, 49.0, -22.0, 3.0])
run("P = (x-1)^2 (x-2)^2", a=[1.0, -6.0, 13.0, -12.0, 4.0])

print("--- roots.txt ---")
print(open("roots.txt").read())

Interval: [-12.0000, 12.0000]
Newton:
    x* = +1.00000000  (3 pasi)
    x* = +2.00000000  (3 pasi)
    x* = +3.00000000  (4 pasi)
Olver:
    x* = +1.00000000  (3 pasi)
    x* = +2.00000000  (3 pasi)
    x* = +3.00000000  (3 pasi)
Comparatie (radacini comune):
    x* ~ +1.000000   Newton: 3 pasi | Olver: 3 pasi
    x* ~ +2.000000   Newton: 3 pasi | Olver: 3 pasi
    x* ~ +3.000000   Newton: 4 pasi | Olver: 3 pasi

Interval: [-2.3095, 2.3095]
Newton:
    x* = -1.00000000  (3 pasi)
    x* = +0.14285714  (3 pasi)
    x* = +0.66666667  (3 pasi)
    x* = +1.50000000  (4 pasi)
Olver:
    x* = -1.00000000  (3 pasi)
    x* = +0.14285714  (3 pasi)
    x* = +0.66666667  (3 pasi)
    x* = +1.50000000  (3 pasi)
Comparatie (radacini comune):
    x* ~ -1.000000   Newton: 3 pasi | Olver: 3 pasi
    x* ~ +0.142857   Newton: 3 pasi | Olver: 3 pasi
    x* ~ +0.666667   Newton: 3 pasi | Olver: 3 pasi
    x* ~ +1.500000   Newton: 4 pasi | Olver: 3 pasi

Interval: [-7.1250, 7.1250]
Newton:
    x* = +0.2500